# Laboratorio 7  
## Ensambles para clasificación: Random Forest y Gradient Boosting

**Asignatura:** Machine Learning  
**Modalidad:** Teórico-práctica  
**Herramienta principal:** Google Colab  
**Objetivo general:** entrenar, evaluar e interpretar modelos de ensamble para resolver un problema de clasificación binaria de abandono de clientes (*churn*).

## Resultados de aprendizaje

Al finalizar este laboratorio, el estudiante será capaz de:

1. cargar y explorar un dataset tabular de clasificación;
2. preparar variables numéricas y categóricas dentro de un *pipeline*;
3. entrenar modelos de **Random Forest** y **Gradient Boosting**;
4. evaluar los modelos con **accuracy**, **precision**, **recall**, **F1-score**, **ROC AUC** y **matriz de confusión**;
5. interpretar qué variables explican en mayor medida la predicción del modelo;
6. comparar resultados y justificar cuál modelo conviene usar.

## Contexto del problema

Una empresa de telecomunicaciones desea anticipar qué clientes podrían abandonar el servicio.  
Cada fila representa un cliente y la variable objetivo es:

- **0** = el cliente no abandona el servicio;
- **1** = el cliente abandona el servicio.

Se cuenta con variables como antigüedad, tipo de contrato, pagos atrasados, llamadas a soporte y nivel de satisfacción.

## ¿Qué aprenderás hoy?

En este laboratorio trabajaremos con modelos de **ensamble**.

### 1) Random Forest
Construye muchos árboles de decisión y combina sus predicciones.  
Sus ventajas principales son:

- reduce el sobreajuste respecto a un árbol individual;
- maneja relaciones no lineales;
- permite estimar importancia de variables;
- funciona muy bien como línea base robusta.

### 2) Gradient Boosting
Construye árboles pequeños de forma secuencial.  
Cada nuevo árbol intenta corregir los errores del anterior.

Sus ventajas principales son:

- suele lograr alto rendimiento predictivo;
- aprende patrones complejos;
- permite un ajuste fino del modelo.

### Idea clave
Un modelo de ensamble busca que la combinación de varios modelos débiles produzca una predicción más estable y precisa.

## Instrucciones

1. Sube a Colab uno de los datasets del laboratorio:
   - `dataset_seccion_1.csv`
   - `dataset_seccion_2.csv`
   - `dataset_seccion_3.csv`
   - `dataset_seccion_4.csv`

2. Ejecuta las celdas en orden.
3. Analiza las métricas.
4. Responde las preguntas finales.

In [ ]:
# Librerías: ensambles (RF / Gradient Boosting) y métricas
# ============================
# 1. Librerías
# ============================
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# from google.colab import files  # solo Colab

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)
from sklearn.inspection import permutation_importance

plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True

## Paso 1. Cargar el dataset

Ejecuta la siguiente celda, selecciona el archivo CSV y verifica que las columnas se hayan cargado correctamente.

In [ ]:
# Carga el dataset de la sección elegida
# ============================
# 2. Carga del dataset
# ============================
# Cambia el número de sección (1 a 4) según tu caso.
df = pd.read_csv("../datasets/lab07/dataset_seccion_1.csv")

print("Dimensiones:", df.shape)
df.head()

## Paso 2. Exploración inicial

Antes de entrenar un modelo, siempre debemos revisar:

- estructura del dataset;
- tipos de datos;
- valores faltantes;
- balance de clases;
- estadísticos básicos.

In [ ]:
# Exploración: tipos, faltantes y balance de clases
# ============================
# 3. Exploración inicial
# ============================
print(df.info())
print("\nValores faltantes por columna:")
print(df.isna().sum())

print("\nDistribución de la variable objetivo:")
print(df["churn"].value_counts())
print("\nDistribución porcentual:")
print(df["churn"].value_counts(normalize=True).round(3))

df.describe(include="all").T

In [ ]:
# Gráfico del balance de clases
# Gráfico de balance de clases
class_counts = df["churn"].value_counts().sort_index()

plt.figure(figsize=(6,4))
plt.bar(["No churn", "Churn"], class_counts.values)
plt.title("Distribución de clases")
plt.ylabel("Número de clientes")
plt.show()

## Paso 3. Separar variables predictoras y variable objetivo

En este problema:

- **X** contiene las variables de entrada;
- **y** contiene la variable objetivo `churn`.

No usaremos `customer_id` porque es un identificador, no una característica útil para aprender patrones.

In [ ]:
# Define X (predictoras) e y (churn); separa num/categóricas
# ============================
# 4. Definir X e y
# ============================
X = df.drop(columns=["customer_id", "churn"])
y = df["churn"]

categorical_features = ["promotion_used", "contract_type"]
numeric_features = [col for col in X.columns if col not in categorical_features]

print("Variables numéricas:", numeric_features)
print("Variables categóricas:", categorical_features)

## Paso 4. División entrenamiento/prueba

Separaremos el dataset en:

- **75% entrenamiento**
- **25% prueba**

Usamos `stratify=y` para conservar la proporción de clases en ambos subconjuntos.

In [ ]:
# Split 75/25 estratificado
# ============================
# 5. Train-test split
# ============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

## Paso 5. Preprocesamiento con `ColumnTransformer`

### ¿Por qué usar un pipeline?
Porque evita errores, organiza el flujo de trabajo y permite que el mismo tratamiento se aplique tanto al conjunto de entrenamiento como al de prueba.

### Tratamiento aplicado
- Variables numéricas: imputación por mediana.
- Variables categóricas: imputación por valor más frecuente + codificación one-hot.

In [ ]:
# Preprocesa: imputa numéricas, imputa+one-hot categóricas
# ============================
# 6. Preprocesamiento
# ============================
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

## Paso 6. Construcción de modelos

Trabajaremos con dos modelos:

### Random Forest
Parámetros usados:
- `n_estimators=300`: cantidad de árboles.
- `max_depth=8`: profundidad máxima para evitar sobreajuste extremo.
- `class_weight="balanced"`: corrige parcialmente el desbalance de clases.

### Gradient Boosting
Parámetros usados:
- `n_estimators=200`
- `learning_rate=0.05`
- `max_depth=3` en cada árbol base

In [ ]:
# Define Random Forest y Gradient Boosting en pipeline
# ============================
# 7. Modelos
# ============================
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        random_state=42,
        class_weight="balanced"
    ))
])

gb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        random_state=42
    ))
])

## Paso 7. Entrenamiento

En esta etapa el modelo aprende patrones a partir de los datos de entrenamiento.

In [ ]:
# Entrena ambos modelos
# ============================
# 8. Entrenamiento
# ============================
rf_model.fit(X_train, y_train)
gb_model.fit(X_train, y_train)

print("Entrenamiento completado.")

## Paso 8. Predicciones y probabilidades

- `predict()` devuelve la clase estimada.
- `predict_proba()` devuelve la probabilidad estimada para cada clase.

Para ROC AUC necesitamos probabilidades, no solo clases predichas.

In [ ]:
# Predicciones (clase) y probabilidades (para ROC AUC)
# ============================
# 9. Predicciones
# ============================
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

gb_pred = gb_model.predict(X_test)
gb_prob = gb_model.predict_proba(X_test)[:, 1]

## Paso 9. Función de evaluación

Crearemos una función para calcular métricas de forma ordenada.

In [ ]:
# Función de evaluación: calcula todas las métricas
# ============================
# 10. Función de evaluación
# ============================
def evaluate_model(y_true, y_pred, y_prob):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_prob)
    }

rf_metrics = evaluate_model(y_test, rf_pred, rf_prob)
gb_metrics = evaluate_model(y_test, gb_pred, gb_prob)

metrics_df = pd.DataFrame([rf_metrics, gb_metrics], index=["Random Forest", "Gradient Boosting"])
metrics_df.round(4)

## Interpretación de métricas

### Accuracy
Porcentaje total de predicciones correctas.

### Precision
De todos los clientes que el modelo marcó como abandono, cuántos realmente abandonaban.

### Recall
De todos los clientes que realmente abandonaban, cuántos fueron detectados.

### F1-score
Promedio armónico entre precision y recall.  
Es muy útil cuando hay desbalance de clases.

### ROC AUC
Mide la capacidad del modelo para separar ambas clases usando probabilidades.

In [ ]:
# Reporte de clasificación de cada modelo
# ============================
# 11. Reporte de clasificación
# ============================
print("=== RANDOM FOREST ===")
print(classification_report(y_test, rf_pred))

print("\n=== GRADIENT BOOSTING ===")
print(classification_report(y_test, gb_pred))

## Paso 10. Matriz de confusión

La matriz de confusión responde cuatro preguntas:

- **TN**: ¿cuántos no abandono fueron bien clasificados?
- **FP**: ¿cuántos no abandono fueron clasificados erróneamente como abandono?
- **FN**: ¿cuántos abandono no fueron detectados?
- **TP**: ¿cuántos abandono fueron detectados correctamente?

In [ ]:
# Matrices de confusión lado a lado
# ============================
# 12. Matrices de confusión
# ============================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, rf_pred,
    display_labels=["No churn", "Churn"],
    colorbar=False,
    ax=axes[0]
)
axes[0].set_title("Random Forest")

ConfusionMatrixDisplay.from_predictions(
    y_test, gb_pred,
    display_labels=["No churn", "Churn"],
    colorbar=False,
    ax=axes[1]
)
axes[1].set_title("Gradient Boosting")

plt.tight_layout()
plt.show()

## Paso 11. Curva ROC

La curva ROC compara:

- **TPR** (*True Positive Rate*)
- **FPR** (*False Positive Rate*)

Mientras más cerca esté la curva de la esquina superior izquierda, mejor suele ser el modelo.

In [ ]:
# Curvas ROC comparadas
# ============================
# 13. Curvas ROC
# ============================
fig, ax = plt.subplots(figsize=(7, 5))

RocCurveDisplay.from_predictions(y_test, rf_prob, ax=ax, name="Random Forest")
RocCurveDisplay.from_predictions(y_test, gb_prob, ax=ax, name="Gradient Boosting")

plt.title("Comparación de curvas ROC")
plt.show()

## Paso 12. Importancia de variables

En Random Forest podemos analizar qué variables contribuyen más a la predicción.

Aquí usaremos **importancia por permutación**, una técnica robusta porque mide cuánto cae el desempeño cuando desordenamos una variable.

In [ ]:
# Importancia por permutación (qué variables pesan más)
# ============================
# 14. Importancia por permutación
# ============================
perm = permutation_importance(
    rf_model, X_test, y_test,
    n_repeats=15,
    random_state=42,
    scoring="f1"
)

importance_df = pd.DataFrame({
    "variable": X_test.columns,
    "importancia": perm.importances_mean
}).sort_values("importancia", ascending=False)

importance_df

In [ ]:
# Gráfico de importancia de variables
plt.figure(figsize=(8, 5))
plt.barh(importance_df["variable"][::-1], importance_df["importancia"][::-1])
plt.title("Importancia de variables - Random Forest")
plt.xlabel("Disminución promedio del F1")
plt.show()

## Paso 13. Conclusión automática del notebook

La siguiente celda genera un resumen simple para apoyar la interpretación.

In [ ]:
# Resumen automático: mejor modelo y variables clave
best_model = metrics_df["f1"].idxmax()
best_f1 = metrics_df["f1"].max()
best_auc = metrics_df.loc[best_model, "roc_auc"]

print(f"El modelo con mejor F1-score fue: {best_model}")
print(f"F1-score: {best_f1:.4f}")
print(f"ROC AUC : {best_auc:.4f}")

print("\nVariables más influyentes según Random Forest:")
print(importance_df.head(5).to_string(index=False))

## Actividad del estudiante

Responde en tu informe:

1. ¿Qué modelo obtuvo mejor **F1-score**?
2. ¿Qué modelo obtuvo mejor **ROC AUC**?
3. ¿Qué variables aparecen como más importantes?
4. ¿Por qué no basta con observar solo el **accuracy**?
5. Si el costo de no detectar un cliente que abandonará es alto, ¿qué métrica te interesa más: **precision** o **recall**? Justifica.

### Respuestas — Actividad del estudiante

1. **Mejor F1-score:** lo indica la celda de resumen (`best_model`). En general el Gradient Boosting suele igualar o superar levemente al Random Forest en F1, aunque depende de la sección; se reporta el que muestre el mayor F1 en la tabla de métricas.
2. **Mejor ROC AUC:** se lee de la columna `roc_auc` de la tabla; mide la capacidad de separar clases por probabilidad. Suele ser muy parecido entre ambos modelos.
3. **Variables más importantes:** las que aparecen arriba en la importancia por permutación (típicamente antigüedad/`tenure`, cargos mensuales, llamadas a soporte, atrasos de pago y satisfacción).
4. **Por qué no basta el accuracy:** con clases desbalanceadas, un modelo puede tener accuracy alto prediciendo casi siempre "no churn" sin detectar a los que abandonan. Por eso se revisan recall, F1 y ROC AUC.
5. **Precision o recall si el costo de no detectar es alto:** interesa el **recall**, porque mide cuántos clientes que sí abandonan logramos detectar. Un recall bajo significa dejar pasar abandonos (el error caro); aceptamos más falsas alarmas (menor precision) con tal de no perder clientes en riesgo.


## Desafío adicional

Realiza al menos dos de las siguientes mejoras:

- ajustar hiperparámetros del Random Forest;
- ajustar hiperparámetros del Gradient Boosting;
- cambiar el umbral de decisión de 0.50 a 0.40 o 0.60;
- agregar validación cruzada;
- comparar con un Árbol de Decisión simple;
- analizar qué ocurre si eliminas una de las variables importantes.

## Exportar resultados

La siguiente celda guarda las predicciones del mejor modelo en un archivo CSV, lo cual facilita la entrega del laboratorio.

In [ ]:
# Exporta las predicciones del mejor modelo a CSV
# ============================
# 15. Exportar predicciones
# ============================
if best_model == "Random Forest":
    final_pred = rf_pred
    final_prob = rf_prob
else:
    final_pred = gb_pred
    final_prob = gb_prob

results = X_test.copy()
results["y_true"] = y_test.values
results["y_pred"] = final_pred
results["prob_churn"] = final_prob

output_name = "predicciones_laboratorio_7.csv"
results.to_csv(output_name, index=False)

# from google.colab import files  # solo Colab
# files.download(output_name)

results.head()

## Cierre

En este laboratorio aprendiste que:

- los modelos de ensamble suelen mejorar el rendimiento respecto a modelos individuales;
- la evaluación correcta requiere varias métricas, no solo *accuracy*;
- la interpretación de variables es clave para tomar decisiones de negocio;
- un buen flujo de trabajo en machine learning debe integrar **carga, exploración, preparación, entrenamiento, evaluación e interpretación**.